# SQL Database Generation Project

This notebook generates a synthetic SQLite database for an E-Commerce system as part of a Data Science SQL assignment.

The goal is to create a realistic relational database containing multiple tables, foreign keys, and composite keys. The data is randomly generated using Python and the Faker library to simulate real-world customer and transaction data.

The generated database will include:
- Multiple relational tables
- At least 1000 rows of data
- Different data types (nominal, ordinal, interval, ratio)
- Foreign key relationships between tables
- A composite primary key
- Intentionally introduced missing and duplicate values to simulate realistic datasets

## 1. Import Required Libraries

In [ ]:
!pip install faker -q

import random
import sqlite3
from datetime import datetime, timedelta, date

import pandas as pd
from faker import Faker

# Reproducibility (professional)
SEED = 42
random.seed(SEED)

fake = Faker("en_GB")
Faker.seed(SEED)

## 2. Define Value Pools and Helper Functions
We define realistic category values (nominal/ordinal) and helper functions used to generate synthetic but realistic data.

In [ ]:
random.seed(42)

# Nominal
FIRST_NAMES = ["Aisha","Daniel","Chinedu","Emily","Grace","Hannah","Ibrahim","James","Kemi","Liam",
               "Mary","Noah","Olivia","Peter","Queen","Ruth","Samuel","Tolu","Uche","Zara"]
LAST_NAMES  = ["Adeyemi","Brown","Cole","Davis","Evans","Garcia","Hughes","Ibrahim","Johnson","Khan",
               "Lee","Miller","Nwosu","Ojo","Okafor","Patel","Smith","Taylor","Williams","Young"]

COUNTRIES = ["UK","Nigeria","Ghana","Kenya","USA","Canada","Germany","France","Spain","Netherlands"]
GENDERS = ["Male","Female","Other","Prefer not to say"]
SHIPPING_METHODS = ["Standard","Express","Next-Day","Click&Collect"]
CATEGORIES = ["Electronics","Fashion","Home","Beauty","Sports","Books","Groceries"]
BRANDS = ["Nova","Apex","Zenith","Pulse","Orion","UrbanCo","Bright","Nimbus","PrimeWare","Echo"]

# Ordinal
LOYALTY_TIERS = ["Bronze","Silver","Gold","Platinum"]
PRIORITY_LEVELS = ["Low", "Medium", "High", "Urgent"]

ORDER_STATUS  = ["Placed","Shipped","Delivered","Cancelled","Returned"]

REVIEW_TEXTS = [
    "Good quality, would buy again.",
    "Packaging was great and delivery was fast.",
    "Not as expected, but okay for the price.",
    "Excellent product, highly recommended.",
    "Average experience. Could be improved.",
    "Very satisfied with the purchase."
]

def rand_date(start: date, end: date) -> date:
    """Random date between start and end."""
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

def make_email(first, last):
    """Create a plausible email address."""
    domain = random.choice(["gmail.com","yahoo.com","outlook.com","hotmail.com"])
    num = str(random.randint(1, 9999))
    return f"{first.lower()}.{last.lower()}{num}@{domain}"

def weighted_choice(items, weights):
    """Pick one item using weights."""
    return random.choices(items, weights=weights, k=1)[0]

def clamp(x, lo, hi):
    """Clamp numeric value to [lo, hi]."""
    return max(lo, min(hi, x))

## 3. Create SQLite Database Connection

This section creates the SQLite database that will store the generated tables and data.  
Foreign key enforcement is enabled to ensure relational integrity between tables.

In [ ]:
# Database file name
db_path = "ecommerce_project.db"

# This Create SQLite database connection
conn = sqlite3.connect(db_path)

# Enable foreign key constraints
conn.execute("PRAGMA foreign_keys = ON;")

# Create cursor object to execute SQL commands
cur = conn.cursor()

print("SQLite database successfully created at:", db_path)

SQLite database successfully created at: ecommerce_project.db


## 4. Reset Database Tables

Before creating new tables, any existing tables are dropped.  
This ensures the notebook can be rerun without conflicts or duplicate structures.

In [ ]:
# Drop tables if they already exist to allow clean reruns of the notebook
cur.executescript("""
DROP TABLE IF EXISTS reviews;
DROP TABLE IF EXISTS order_items;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS customers;
""")

conn.commit()

print("Old tables dropped (if they existed).")

Old tables dropped (if they existed).


## 5. Define Database Schema (Table Structure)

This section defines the relational database schema for the e-commerce system using SQL.

Five tables are created:

• **customers** – stores customer information and demographic attributes.  
• **products** – stores product catalogue details.  
• **orders** – records customer purchases and order-level information.  
• **order_items** – a bridge table that links orders and products and records the quantity purchased.  
• **reviews** – stores product reviews submitted by customers.

The schema includes several database constraints to ensure data integrity:

- **Primary Keys** to uniquely identify each record.
- **Foreign Keys** to establish relationships between tables.
- **Composite Primary Key** in the `order_items` table (`order_id`, `product_id`).
- **CHECK constraints** to restrict values to realistic ranges.
- **UNIQUE constraint** on customer email addresses.
- **NOT NULL constraints** to enforce required attributes.

These constraints help ensure that the database maintains **referential integrity and realistic data structure**, which is essential for reliable data analysis.

In [ ]:
cur.executescript("""
-- CUSTOMERS (1000+ rows target)
CREATE TABLE customers (
    customer_id      INTEGER PRIMARY KEY,
    full_name        TEXT NOT NULL,
    email            TEXT UNIQUE,                 -- UNIQUE but allows NULL (realistic missing emails)
    gender           TEXT NOT NULL CHECK (gender IN ('Male','Female','Other','Prefer not to say')),
    country          TEXT NOT NULL,
    signup_date      TEXT NOT NULL,               -- stored as ISO date string (interval-type)
    loyalty_tier     TEXT NOT NULL CHECK (loyalty_tier IN ('Bronze','Silver','Gold','Platinum')),
    age              INTEGER NOT NULL CHECK (age BETWEEN 16 AND 90)  -- ratio
);

-- PRODUCTS
CREATE TABLE products (
    product_id       INTEGER PRIMARY KEY,
    product_name     TEXT NOT NULL,
    category         TEXT NOT NULL,
    brand            TEXT NOT NULL,
    unit_price       REAL NOT NULL CHECK (unit_price >= 0),          -- ratio
    stock_qty        INTEGER NOT NULL CHECK (stock_qty >= 0),        -- ratio
    created_date     TEXT NOT NULL                                   -- interval-type
);

-- ORDERS
CREATE TABLE orders (
    order_id         INTEGER PRIMARY KEY,
    customer_id      INTEGER NOT NULL,
    order_date       TEXT NOT NULL,
    shipping_method  TEXT NOT NULL CHECK (shipping_method IN ('Standard','Express','Next-Day','Click&Collect')),
    order_status     TEXT NOT NULL CHECK (order_status IN ('Placed','Shipped','Delivered','Cancelled','Returned')),
    delivery_temp_c  REAL CHECK (delivery_temp_c BETWEEN -10 AND 40),
    shipping_fee     REAL NOT NULL CHECK (shipping_fee >= 0),        -- ratio
    discount_amount  REAL CHECK (discount_amount IS NULL OR discount_amount >= 0), -- ratio + missing
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

-- ORDER ITEMS (bridge table)
-- Composite Primary Key = (order_id, product_id) for Excellent schema links
CREATE TABLE order_items (
    order_id               INTEGER NOT NULL,
    product_id             INTEGER NOT NULL,
    quantity               INTEGER NOT NULL CHECK (quantity > 0),     -- ratio
    unit_price_at_purchase REAL NOT NULL CHECK (unit_price_at_purchase >= 0),
    line_total             REAL NOT NULL CHECK (line_total >= 0),
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id) REFERENCES orders(order_id) ON DELETE CASCADE,
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);

-- REVIEWS (adds realism & joins)
CREATE TABLE reviews (
    review_id        INTEGER PRIMARY KEY,
    customer_id      INTEGER NOT NULL,
    product_id       INTEGER NOT NULL,
    review_score     INTEGER NOT NULL CHECK (review_score BETWEEN 1 AND 5),  -- ordinal
    review_text      TEXT,                                                  -- missing allowed
    review_date      TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")
conn.commit()

print("Tables created with constraints + foreign keys + composite key.")

Tables created with constraints + foreign keys + composite key.


## 6. Generate and Insert Synthetic Customer Data (1200 rows)

This section generates a realistic **customers** dataset and inserts it into the SQLite database.

To improve realism (as required by the rubric), the synthetic data intentionally includes:
- **Duplicate values**: ~3% of customer names are deliberately duplicated.
- **Missing values**: ~5% of customer emails are set to NULL to simulate incomplete records.

The data is also generated with realistic distributions:
- Countries and loyalty tiers are **weighted** to reflect common real-world patterns.
- Age values follow a **normal distribution** (mean ≈ 31) and are clamped to valid bounds (16–90).

In [ ]:
N_CUSTOMERS = 1200  # >= 1000 (requirement)

customers = []
used_emails = set()

start_signup = date(2022, 1, 1)
end_signup   = date(2026, 2, 1)

for cid in range(1, N_CUSTOMERS + 1):
    first = random.choice(FIRST_NAMES)
    last  = random.choice(LAST_NAMES)

    # DELIBERATE DUPLICATES (Rubric realism):
    # 3% chance we intentionally reuse an existing full_name
    if customers and random.random() < 0.03:
        full_name = random.choice(customers)[1]  # reuse existing name (duplicate)
    else:
        full_name = f"{first} {last}"

    gender = weighted_choice(GENDERS, [0.48, 0.48, 0.02, 0.02])
    country = weighted_choice(COUNTRIES, [0.40, 0.25, 0.06, 0.06, 0.08, 0.05, 0.03, 0.03, 0.03, 0.01])

    # loyalty tier (ordinal) - realistic skew towards Bronze/Silver
    loyalty = weighted_choice(LOYALTY_TIERS, [0.55, 0.30, 0.12, 0.03])

    signup = rand_date(start_signup, end_signup).isoformat()

    # age: realistic
    age = int(clamp(random.gauss(31, 10), 16, 90))

    # DELIBERATE MISSING VALUES (Rubric realism):
    # 5% emails intentionally set to NULL
    if random.random() < 0.05:
        email = None
    else:
        # ensure unique emails when not missing
        email = make_email(first, last)
        while email in used_emails:
            email = make_email(first, last)
        used_emails.add(email)

    customers.append((cid, full_name, email, gender, country, signup, loyalty, age))

cur.executemany("""
INSERT INTO customers(customer_id, full_name, email, gender, country, signup_date, loyalty_tier, age)
VALUES (?, ?, ?, ?, ?, ?, ?, ?);
""", customers)

conn.commit()
print("Inserted customers:", len(customers))

Inserted customers: 1200


## 7. Generate and Insert Synthetic Product Data (250 rows)

This section generates a realistic **product catalogue** and inserts it into the `products` table.

Realism choices:
- Product names are constructed from a **brand + adjective + item type**.
- Prices follow a **skewed distribution** (more low–mid prices and fewer high prices), with Electronics generally more expensive.
- Stock quantities are generated using a normal distribution and clamped to realistic limits.

In [ ]:
N_PRODUCTS = 250

products = []
start_prod = date(2021, 1, 1)
end_prod   = date(2026, 2, 1)

for pid in range(1, N_PRODUCTS + 1):
    category = random.choice(CATEGORIES)
    brand = random.choice(BRANDS)

    # product name: looks realistic
    adjective = random.choice(["Pro","Lite","Max","Mini","Smart","Eco","Ultra","Flex","Air"])
    item = random.choice(["Headphones","T-Shirt","Blender","Moisturiser","Football","Novel","Coffee","Backpack","Sneakers","Keyboard"])
    product_name = f"{brand} {adjective} {item}"

    # price: skewed distribution
    base = abs(random.gauss(35, 25))
    if category == "Electronics":
        base += abs(random.gauss(80, 60))
    unit_price = round(clamp(base, 2, 1200), 2)

    stock_qty = int(clamp(random.gauss(120, 80), 0, 1000))

    created = rand_date(start_prod, end_prod).isoformat()

    products.append((pid, product_name, category, brand, unit_price, stock_qty, created))

cur.executemany("""
INSERT INTO products(product_id, product_name, category, brand, unit_price, stock_qty, created_date)
VALUES (?, ?, ?, ?, ?, ?, ?);
""", products)

conn.commit()
print("Inserted products:", len(products))

Inserted products: 250


## 8. Generate and Insert Synthetic Order Data (3,200 rows)

This section generates realistic order records and inserts them into the `orders` table.

Realism choices:
- Each order is linked to an existing customer (`customer_id`) to maintain referential integrity.
- Shipping method and order status are generated using **weighted probabilities** (e.g., most orders are Delivered).
- Shipping fees are generated from a realistic distribution and clamped to valid bounds.
- **Missing values** are deliberately introduced: ~20% of orders have a NULL `discount_amount` to simulate incomplete or non-discounted transactions.

In [ ]:
N_ORDERS = 3200

orders = []
start_order = date(2023, 1, 1)
end_order   = date(2026, 2, 1)

for oid in range(1, N_ORDERS + 1):
    customer_id = random.randint(1, N_CUSTOMERS)
    order_date = rand_date(start_order, end_order).isoformat()

    shipping_method = weighted_choice(SHIPPING_METHODS, [0.60, 0.25, 0.10, 0.05])

    # status skew: most delivered
    order_status = weighted_choice(ORDER_STATUS, [0.10, 0.15, 0.65, 0.06, 0.04])

    shipping_fee = round(clamp(abs(random.gauss(4.5, 2.0)), 0, 25), 2)
    delivery_temp_c = round(clamp(random.gauss(12, 7), -10, 40), 1)

    # DELIBERATE MISSING: 20% of orders have NULL discount_amount
    if random.random() < 0.20:
        discount = None
    else:
        discount = round(clamp(abs(random.gauss(3.0, 4.0)), 0, 50), 2)

    delivery_temp_c = round(clamp(random.gauss(12, 7), -10, 40), 1)
    orders.append((oid, customer_id, order_date, shipping_method, order_status, delivery_temp_c, shipping_fee, discount))

cur.executemany("""
INSERT INTO orders(order_id, customer_id, order_date, shipping_method, order_status, delivery_temp_c, shipping_fee, discount_amount)
VALUES (?, ?, ?, ?, ?, ?, ?, ?);
""", orders)

conn.commit()
print("Inserted orders:", len(orders))

Inserted orders: 3200


## 9. Generate and Insert Order Items (Bridge Table with Composite Key)

This section populates the `order_items` table, which links orders to products (many-to-many relationship).

Key design features:
- `order_items` uses a **composite primary key**: (`order_id`, `product_id`).
- The code ensures each (`order_id`, `product_id`) pair is unique to avoid violating the composite key.
- Product prices are read from the `products` table to generate realistic `unit_price_at_purchase`.
- Purchase prices vary slightly (±15%) to simulate promotions and price changes over time.
- `line_total` is calculated as `quantity × unit_price_at_purchase` to ensure consistency.

In [ ]:


# pull product prices for realistic unit_price_at_purchase (if not already done)
prod_df = pd.read_sql_query("SELECT product_id, unit_price FROM products;", conn)
price_map = dict(zip(prod_df["product_id"], prod_df["unit_price"]))

order_items = []
used_pairs = set()

attempts = 0
target_rows = 6000

while len(order_items) < target_rows and attempts < 100000:
    attempts += 1
    order_id = random.randint(1, N_ORDERS)
    product_id = random.randint(1, N_PRODUCTS)

    # skip duplicates of composite key
    if (order_id, product_id) in used_pairs:
        continue

    used_pairs.add((order_id, product_id))

    quantity = weighted_choice([1,2,3,4,5,6], [0.55,0.20,0.12,0.07,0.04,0.02])
    base_price = price_map[product_id]
    unit_price_at_purchase = round(clamp(base_price * random.uniform(0.85, 1.05), 0.5, 5000), 2)
    line_total = round(unit_price_at_purchase * quantity, 2)

    order_items.append((order_id, product_id, quantity, unit_price_at_purchase, line_total))

cur.executemany("""
INSERT INTO order_items(order_id, product_id, quantity, unit_price_at_purchase, line_total)
VALUES (?, ?, ?, ?, ?);
""", order_items)

conn.commit()
print("Inserted order_items:", len(order_items), "| Attempts:", attempts)

Inserted order_items: 6000 | Attempts: 6022


## 10. Generate and Insert Product Reviews (1,800 rows)

This section generates synthetic product reviews and inserts them into the `reviews` table.

Realism choices:
- Review scores (1–5) are treated as an **ordinal** attribute.
- Review text is optional: ~40% of reviews have NULL `review_text` to reflect cases where customers leave only a rating.
- Each review references valid customers and products to maintain referential integrity.

In [ ]:
N_REVIEWS = 1800
reviews = []

start_review = date(2023, 1, 1)
end_review   = date(2026, 2, 1)

for rid in range(1, N_REVIEWS + 1):
    customer_id = random.randint(1, N_CUSTOMERS)
    product_id = random.randint(1, N_PRODUCTS)

    # ordinal score
    score = weighted_choice([1,2,3,4,5], [0.05,0.10,0.25,0.35,0.25])

    # DELIBERATE MISSING: 40% have no review_text
    if random.random() < 0.40:
        text = None
    else:
        text = random.choice(REVIEW_TEXTS)

    rdate = rand_date(start_review, end_review).isoformat()

    reviews.append((rid, customer_id, product_id, score, text, rdate))

cur.executemany("""
INSERT INTO reviews(review_id, customer_id, product_id, review_score, review_text, review_date)
VALUES (?, ?, ?, ?, ?, ?);
""", reviews)

conn.commit()
print("Inserted reviews:", len(reviews))

Inserted reviews: 1800


## 11. Validate Database Population (Row Counts)

This section checks that each table has been successfully populated and confirms that the database meets the assignment requirement of at least **1000+ rows** in at least one table.

In [ ]:

print("=" * 40)
print("ROW COUNTS")
print("=" * 40)

for table in ["customers", "products", "orders", "order_items", "reviews"]:
    n = cur.execute(f"SELECT COUNT(*) FROM {table};").fetchone()[0]
    print(f"  {table:<15} {n:>6} rows")

print("=" * 40)

ROW COUNTS
  customers         1200 rows
  products           250 rows
  orders            3200 rows
  order_items       6000 rows
  reviews           1800 rows


## 12. Data Realism Checks (Missing Values and Duplicates)

This section verifies that the synthetic dataset includes realistic imperfections required by the rubric:
- Missing values (NULLs) in selected attributes
- Duplicate values in selected attributes

The results provide evidence that missingness and duplication were introduced deliberately and in realistic proportions.

In [ ]:

print("=" * 40)
print("DATA REALISM CHECKS")
print("=" * 40)

# Total rows in tables
total_customers = cur.execute("SELECT COUNT(*) FROM customers;").fetchone()[0]
total_orders = cur.execute("SELECT COUNT(*) FROM orders;").fetchone()[0]
total_reviews = cur.execute("SELECT COUNT(*) FROM reviews;").fetchone()[0]

# Missing emails
missing_emails = cur.execute(
    "SELECT COUNT(*) FROM customers WHERE email IS NULL;"
).fetchone()[0]

print(f"Missing emails: {missing_emails} "
      f"({round(missing_emails/total_customers*100,2)}%)")

# Missing discounts
missing_discounts = cur.execute(
    "SELECT COUNT(*) FROM orders WHERE discount_amount IS NULL;"
).fetchone()[0]

print(f"Missing discounts: {missing_discounts} "
      f"({round(missing_discounts/total_orders*100,2)}%)")

# Missing review text
missing_reviews = cur.execute(
    "SELECT COUNT(*) FROM reviews WHERE review_text IS NULL;"
).fetchone()[0]

print(f"Missing review_text: {missing_reviews} "
      f"({round(missing_reviews/total_reviews*100,2)}%)")

print("\n" + "=" * 40)
print("Duplicate customer names (deliberate)")
print("=" * 40)

dup_names = pd.read_sql_query("""
SELECT full_name, COUNT(*) AS count
FROM customers
GROUP BY full_name
HAVING COUNT(*) > 1
ORDER BY count DESC
LIMIT 10;
""", conn)

dup_names

DATA REALISM CHECKS
Missing emails: 52 (4.33%)
Missing discounts: 620 (19.38%)
Missing review_text: 698 (38.78%)

Duplicate customer names (deliberate)


,full_name,count
0,Tolu Hughes,9
1,Kemi Taylor,9
2,Zara Young,8
3,James Johnson,8
4,Ibrahim Lee,8
5,Emily Nwosu,8
6,Uche Smith,7
7,Ruth Brown,7
8,Olivia Young,7
9,Mary Miller,7


## 13. Display Database Schema

This section prints the structure of the database, including all tables and their attributes.  
It shows column names, data types, and key constraints such as primary keys and NOT NULL requirements.

This helps verify that the database schema has been created correctly and matches the design described in the report.

In [ ]:

print("=" * 50)
print("DATABASE SCHEMA")
print("=" * 50)

# Get list of tables
tables = cur.execute("""
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
""").fetchall()

for (table_name,) in tables:
    print(f"\nTABLE: {table_name}")

    cols = cur.execute(f"PRAGMA table_info({table_name});").fetchall()

    for col in cols:
        col_id, name, dtype, notnull, default, pk = col

        flags = []
        if pk:
            flags.append("PRIMARY KEY")
        if notnull:
            flags.append("NOT NULL")
        if default is not None:
            flags.append(f"DEFAULT {default}")

        flag_str = ", ".join(flags)

        print(f"   {name:<25} {dtype:<10} {flag_str}")

print("\n" + "=" * 50)

DATABASE SCHEMA

TABLE: customers
   customer_id               INTEGER    PRIMARY KEY
   full_name                 TEXT       NOT NULL
   email                     TEXT       
   gender                    TEXT       NOT NULL
   country                   TEXT       NOT NULL
   signup_date               TEXT       NOT NULL
   loyalty_tier              TEXT       NOT NULL
   age                       INTEGER    NOT NULL

TABLE: order_items
   order_id                  INTEGER    PRIMARY KEY, NOT NULL
   product_id                INTEGER    PRIMARY KEY, NOT NULL
   quantity                  INTEGER    NOT NULL
   unit_price_at_purchase    REAL       NOT NULL
   line_total                REAL       NOT NULL

TABLE: orders
   order_id                  INTEGER    PRIMARY KEY
   customer_id               INTEGER    NOT NULL
   order_date                TEXT       NOT NULL
   shipping_method           TEXT       NOT NULL
   order_status              TEXT       NOT NULL
   delivery_temp_c       

## 14. Example Analytical Queries

This section demonstrates how the relational database can be queried to extract meaningful insights.

The examples below illustrate:
- Joining multiple tables using foreign key relationships
- Aggregating data using SQL functions such as COUNT, SUM, and AVG
- Performing simple analytical queries on the generated dataset

In [ ]:
from IPython.display import display

# Example 1: Top customers by number of orders (table link: customers -> orders)
q1 = pd.read_sql_query("""
SELECT c.customer_id, c.full_name, c.loyalty_tier, COUNT(o.order_id) AS n_orders
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.full_name, c.loyalty_tier
ORDER BY n_orders DESC
LIMIT 10;
""", conn)
print("Top 10 customers by order count:")
display(q1)

# Example 2: Revenue by category (table link: order_items -> products)
q2 = pd.read_sql_query("""
SELECT p.category, ROUND(SUM(oi.line_total), 2) AS revenue
FROM order_items oi
JOIN products p ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC;
""", conn)
print("Revenue by product category:")
display(q2)

# Example 3: Average review score by category (table link: reviews -> products)
q3 = pd.read_sql_query("""
SELECT p.category,
       ROUND(AVG(r.review_score), 2) AS avg_score,
       COUNT(r.review_id) AS total_reviews
FROM reviews r
JOIN products p ON r.product_id = p.product_id
GROUP BY p.category
ORDER BY avg_score DESC;
""", conn)
print("Average review score by category:")
display(q3)

Top 10 customers by order count:


,customer_id,full_name,loyalty_tier,n_orders
0,301,Olivia Okafor,Bronze,9
1,1010,Olivia Nwosu,Silver,9
2,1119,Olivia Hughes,Gold,9
3,657,Zara Young,Bronze,8
4,692,Samuel Miller,Silver,8
5,1123,Zara Smith,Gold,8
6,1170,Ruth Adeyemi,Bronze,8
7,1179,Mary Ojo,Gold,8
8,2,Aisha Cole,Bronze,7
9,60,Aisha Patel,Silver,7


Revenue by product category:


,category,revenue
0,Electronics,199991.18
1,Beauty,61260.53
2,Fashion,58401.64
3,Groceries,56328.29
4,Books,46111.04
5,Sports,44515.77
6,Home,39691.32


Average review score by category:


,category,avg_score,total_reviews
0,Groceries,3.72,279
1,Sports,3.69,210
2,Electronics,3.67,249
3,Fashion,3.65,280
4,Books,3.63,196
5,Beauty,3.62,330
6,Home,3.58,256


## 15. Finalise and Save Database

This step commits any remaining changes to the SQLite database and closes the database connection.  
Closing the connection ensures that all data is safely written to the database file (`ecommerce_project.db`).

In [ ]:
conn.commit()
conn.close()
print("Database saved and connection closed.")

Database saved and connection closed.


In [ ]:
from google.colab import files

files.download("ecommerce_project.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>